In [1]:
import pandas as pd
import numpy as np

1. LOAD DATASET

In [5]:
file_path = "/content/E-Commerce Dataset Cleaning.xlsx"

df = pd.read_excel('/content/E-Commerce Dataset Cleaning.xlsx')

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())



Dataset loaded successfully!
Shape: (67, 12)

Columns:
['Order ID', 'CustomerID', 'Product Name', 'Category', 'Quantity', 'UnitPrice', 'Total Price ', 'Order_Date ', 'Payment_Mode', 'Unnamed: 9', 'Delivery_Status ', 'City or Region ']


2. RAW DATASET INSPECTION

In [6]:
print("\n========== DATASET INSPECTION ==========")

print("\nFirst 5 rows:")
print(df.head())

print("\nDataset information:")
print(df.info())

print("\nStatistical summary:")
print(df.describe(include="all"))

print("\nNumber of rows:", df.shape[0])
print("Number of columns:", df.shape[1])



========== DATASET INSPECTION ==========

First 5 rows:
              Order ID  CustomerID        Product Name  Category  Quantity  \
0  405-8078784-5731545       17850      Wireless Mouse    Sports         6   
1  171-9198151-1101146       17850  Bluetooth Keyboard  Clothing         6   
2  404-0687676-7273146       17850         USB-C Cable    Sports         8   
3  403-9615377-8133951       17850          Power Bank      Toys         6   
4  407-1069790-7240320       17850    Wireless Earbuds    Beauty         6   

   UnitPrice  Total Price  Order_Date  Payment_Mode  Unnamed: 9  \
0       2.55         36.53  2024-11-12  Net Banking         NaN   
1       3.39        232.79  2024-02-09  Net Banking         NaN   
2       2.75        317.02  2024-09-01  Credit Card         NaN   
3       3.39        173.19  2024-04-01          UPI         NaN   
4       3.39        244.80  2024-09-27  Net Banking         NaN   

    Delivery_Status  City or Region   
0       Order Placed            

3. IDENTIFY COLUMN TYPES

In [7]:
print("\n========== COLUMN DATA TYPES ==========")

print(df.dtypes)


========== COLUMN DATA TYPES ==========
Order ID                    object
CustomerID                   int64
Product Name                object
Category                    object
Quantity                     int64
UnitPrice                  float64
Total Price                float64
Order_Date          datetime64[ns]
Payment_Mode                object
Unnamed: 9                 float64
Delivery_Status             object
City or Region              object
dtype: object


4. MISSING VALUE CHECK

In [9]:
print("\n========== MISSING VALUE REPORT ==========")

missing_count = df.isnull().sum()
missing_percentage = (missing_count / len(df)) * 100

missing_report = pd.DataFrame({
    "Column": df.columns,
    "Missing Count": missing_count.values,
    "Missing Percentage": missing_percentage.round(2).values
})
print(missing_report)



========== MISSING VALUE REPORT ==========
              Column  Missing Count  Missing Percentage
0           Order ID              0                 0.0
1         CustomerID              0                 0.0
2       Product Name              0                 0.0
3           Category              0                 0.0
4           Quantity              0                 0.0
5          UnitPrice              0                 0.0
6       Total Price               0                 0.0
7        Order_Date               0                 0.0
8       Payment_Mode              0                 0.0
9         Unnamed: 9             67               100.0
10  Delivery_Status               0                 0.0
11   City or Region               0                 0.0


5. REMOVE COMPLETELY EMPTY COLUMNS

In [10]:
empty_columns = df.columns[df.isnull().all()].tolist()

print("\nCompletely empty columns:")
print(empty_columns)

df = df.dropna(axis=1, how="all")

print("\nColumns after removing empty columns:")
print(df.columns.tolist())



Completely empty columns:
['Unnamed: 9']

Columns after removing empty columns:
['Order ID', 'CustomerID', 'Product Name', 'Category', 'Quantity', 'UnitPrice', 'Total Price ', 'Order_Date ', 'Payment_Mode', 'Delivery_Status ', 'City or Region ']


6. STANDARDIZE COLUMN NAMES

In [11]:
df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
)

print("\n========== STANDARDIZED COLUMN NAMES ==========")
print(df.columns.tolist())


========== STANDARDIZED COLUMN NAMES ==========
['Order_ID', 'CustomerID', 'Product_Name', 'Category', 'Quantity', 'UnitPrice', 'Total_Price', 'Order_Date', 'Payment_Mode', 'Delivery_Status', 'City_or_Region']


7. REMOVE EXTRA SPACES FROM TEXT DATA

In [12]:
text_columns = df.select_dtypes(include=["object"]).columns

for column in text_columns:
    df[column] = df[column].astype(str).str.strip()

print("\nText fields cleaned successfully.")



Text fields cleaned successfully.


8. CHECK DUPLICATE RECORDS

In [13]:
print("\n========== DUPLICATE ANALYSIS ==========")

duplicate_count = df.duplicated().sum()

print("Fully duplicated rows:", duplicate_count)

if duplicate_count > 0:
    print("\nDuplicate records:")
    print(df[df.duplicated(keep=False)])
else:
    print("No fully duplicated records found.")



========== DUPLICATE ANALYSIS ==========
Fully duplicated rows: 0
No fully duplicated records found.


9. CHECK REPEATED ORDER IDs

In [14]:
print("\n========== REPEATED ORDER IDs ==========")

order_counts = df["Order_ID"].value_counts()

repeated_orders = order_counts[order_counts > 1]

print("Repeated Order IDs:")
print(repeated_orders)

if len(repeated_orders) > 0:
    print("\nRecords with repeated Order IDs:")
    print(
        df[df["Order_ID"].isin(repeated_orders.index)]
        .sort_values("Order_ID")
    )


========== REPEATED ORDER IDs ==========
Repeated Order IDs:
Order_ID
404-2262140-4696366    2
403-4367956-2849158    2
Name: count, dtype: int64

Records with repeated Order IDs:
               Order_ID  CustomerID   Product_Name  Category  Quantity  \
37  403-4367956-2849158       12583  Electric Iron  Clothing        24   
38  403-4367956-2849158       12583    Ceiling Fan  Clothing        20   
61  404-2262140-4696366       17850          Jeans    Sports         6   
62  404-2262140-4696366       17850   Casual Shirt    Beauty         6   

    UnitPrice  Total_Price Order_Date Payment_Mode          Delivery_Status  \
37       1.95        95.41 2024-04-21   Debit Card  Second Delivery Attempt   
38       0.85       490.77 2024-08-03          UPI   Final Delivery Attempt   
61       3.39        10.78 2024-08-22  Net Banking             Customs Hold   
62       3.39       342.66 2024-07-24          UPI        Customs Completed   

   City_or_Region  
37          South  
38          

 10. DATA VALIDATION - QUANTITY

In [15]:
print("\n========== QUANTITY VALIDATION ==========")

negative_quantity = df[df["Quantity"] < 0]

print("Negative quantity records:", len(negative_quantity))

if len(negative_quantity) > 0:
    print(negative_quantity)
else:
    print("No negative quantities found.")


========== QUANTITY VALIDATION ==========
Negative quantity records: 0
No negative quantities found.


11. DATA VALIDATION - UNIT PRICE

In [16]:
print("\n========== UNIT PRICE VALIDATION ==========")

invalid_price = df[df["UnitPrice"] <= 0]

print("Zero or negative prices:", len(invalid_price))

if len(invalid_price) > 0:
    print(invalid_price)
else:
    print("No zero or negative prices found.")



========== UNIT PRICE VALIDATION ==========
Zero or negative prices: 0
No zero or negative prices found.


12. ORDER DATE VALIDATION

In [17]:
print("\n========== ORDER DATE VALIDATION ==========")

df["Order_Date"] = pd.to_datetime(
    df["Order_Date"],
    errors="coerce"
)

invalid_dates = df["Order_Date"].isnull().sum()

print("Invalid or missing dates:", invalid_dates)



========== ORDER DATE VALIDATION ==========
Invalid or missing dates: 0


13. CHECK CATEGORY VALUES

In [18]:
print("\n========== CATEGORY VALIDATION ==========")

print("Unique categories:")
print(df["Category"].unique())

print("\nCategory frequency:")
print(df["Category"].value_counts())


========== CATEGORY VALIDATION ==========
Unique categories:
['Sports' 'Clothing' 'Toys' 'Beauty' 'Books' 'Home & Kitchen'
 'Electronics']

Category frequency:
Category
Toys              15
Home & Kitchen    14
Sports             9
Beauty             8
Clothing           7
Books              7
Electronics        7
Name: count, dtype: int64


14. CHECK PAYMENT MODE

In [19]:
print("\n========== PAYMENT MODE VALIDATION ==========")

print("Unique payment modes:")
print(df["Payment_Mode"].unique())

print("\nPayment mode frequency:")
print(df["Payment_Mode"].value_counts())



========== PAYMENT MODE VALIDATION ==========
Unique payment modes:
['Net Banking' 'Credit Card' 'UPI' 'Cash on Delivery' 'Debit Card']

Payment mode frequency:
Payment_Mode
Net Banking         19
Credit Card         14
UPI                 14
Debit Card          11
Cash on Delivery     9
Name: count, dtype: int64


15. CHECK DELIVERY STATUS

In [20]:
print("\n========== DELIVERY STATUS VALIDATION ==========")

print("Unique delivery statuses:")
print(df["Delivery_Status"].unique())

print("\nDelivery status frequency:")
print(df["Delivery_Status"].value_counts())


========== DELIVERY STATUS VALIDATION ==========
Unique delivery statuses:
['Order Placed' 'Order Confirmed' 'Payment Confirmed' 'Processing'
 'Order Processing' 'Preparing for Shipment' 'Packed' 'Package Ready'
 'Awaiting Pickup' 'Pickup Scheduled' 'Picked Up' 'Shipped' 'In Transit'
 'Transit Delayed' 'At Sorting Center' 'At Distribution Center'
 'Arrived at Hub' 'Departed Hub' 'Reached Local Hub' 'At Local Facility'
 'Out for Delivery' 'Delivery Attempted' 'Delivery Rescheduled'
 'Delivery Delayed' 'Delayed Due to Weather' 'Delayed Due to Traffic'
 'Address Verification Required' 'Address Incorrect' 'Address Incomplete'
 'Customer Unavailable' 'Customer Requested Delay'
 'Customer Requested Reschedule' 'Delivery Instructions Received'
 'Delivery Instructions Missing' 'Recipient Not Available'
 'Recipient Refused' 'Delivery Failed' 'Second Delivery Attempt'
 'Final Delivery Attempt' 'Delivered' 'Delivered Successfully'
 'Delivered to Customer' 'Delivered to Neighbor' 'Delivered to Re

16. CHECK CITY / REGION

In [21]:
print("\n========== CITY / REGION ==========")

print("Unique cities/regions:")
print(df["City_or_Region"].unique())




========== CITY / REGION ==========
Unique cities/regions:
['Pune' 'Mumbai' 'Delhi' 'Bengaluru' 'Hyderabad' 'Chennai' 'Kolkata'
 'Ahmedabad' 'Jaipur' 'Nagpur' 'Nashik' 'Surat' 'Indore' 'Bhopal'
 'Lucknow' 'Patna' 'Kochi' 'Chandigarh' 'Noida' 'Gurugram' 'West' 'North'
 'South' 'East' 'Central']


17. FINAL MISSING VALUE CHECK

In [30]:
print("\n========== FINAL MISSING VALUE CHECK ==========")

print(df.isnull().sum())



========== FINAL MISSING VALUE CHECK ==========
Order_ID            0
CustomerID          0
Product_Name        0
Category            0
Quantity            0
UnitPrice           0
Total_Price         0
Order_Date          0
Payment_Mode        0
Delivery_Status     0
City_or_Region      0
Calculated_Total    0
dtype: int64


18. FINAL DUPLICATE CHECK

In [31]:
print("\n========== FINAL DUPLICATE CHECK ==========")

print(
    "Duplicate rows after cleaning:",
    df.duplicated().sum()
)




========== FINAL DUPLICATE CHECK ==========
Duplicate rows after cleaning: 0


19.REMOVE TEMPORARY VALIDATION COLUMNS

In [32]:
df = df.drop(
    columns=["Calculated_Total", "Total_Difference"],
    errors="ignore"
)


20.SAVE CLEANED DATASET

In [33]:
output_file = "E-Commerce_Dataset_Cleaned.xlsx"

df.to_excel(
    output_file,
    index=False
)

print("\n==========================================")
print("DATA CLEANING COMPLETED SUCCESSFULLY")
print("==========================================")
print("Original rows:", len(pd.read_excel(file_path)))
print("Cleaned rows:", len(df))
print("Output file:", output_file)


DATA CLEANING COMPLETED SUCCESSFULLY
Original rows: 67
Cleaned rows: 67
Output file: E-Commerce_Dataset_Cleaned.xlsx


21.CLEANING SUMMARY

In [34]:
print("\n========== CLEANING SUMMARY ==========")

print("✓ Raw dataset inspected")
print("✓ Missing values checked")
print("✓ Empty columns removed")
print("✓ Column names standardized")
print("✓ Text values trimmed")
print("✓ Duplicate records checked")
print("✓ Repeated Order IDs investigated")
print("✓ Quantity validated")
print("✓ Unit Price validated")
print("✓ Order Date validated")
print("✓ Category checked")
print("✓ Payment Mode checked")
print("✓ Delivery Status checked")
print("✓ City/Region checked")
print("✓ Total Amount validated")
print("✓ Cleaned dataset saved separately")


========== CLEANING SUMMARY ==========
✓ Raw dataset inspected
✓ Missing values checked
✓ Empty columns removed
✓ Column names standardized
✓ Text values trimmed
✓ Duplicate records checked
✓ Repeated Order IDs investigated
✓ Quantity validated
✓ Unit Price validated
✓ Order Date validated
✓ Category checked
✓ Payment Mode checked
✓ Delivery Status checked
✓ City/Region checked
✓ Total Amount validated
✓ Cleaned dataset saved separately
